# Change from V2.0:
## Updated Secondary Label detection and allication in labeling stage

In [2]:
# %% [markdown]
# # RQ2 — Step 0: Clone repositories
# Reads a CSV with column `repo_url`, clones/fetches into CLONE_ROOT,
# and writes a manifest with basic metadata.

# %%
from __future__ import annotations
import csv, subprocess, sys, json, time
from pathlib import Path
from typing import Optional, List
from pathlib import Path

# -----------------------------
# Config (edit as needed)
# -----------------------------


# Use a raw string r"..." for Windows paths with spaces
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"   # put your CSV here
CLONE_ROOT   = WORK_ROOT / "clones"         # repos will clone here
MANIFEST_CSV = WORK_ROOT / "clones_manifest.csv"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)


# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# %%
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)

def repo_dir_name_from_url(url: str) -> str:
    # e.g. https://github.com/owner/name(.git) -> owner__name
    base = url.split("//")[-1]
    parts = base.split("/")
    if len(parts) >= 3:
        owner = parts[-2]
        name  = parts[-1].replace(".git", "")
        return f"{owner}__{name}"
    return base.replace("/", "__").replace(".git", "")

def ensure_cloned(url: str, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)
    if d.exists() and (d / ".git").exists():
        # Refresh remote info (best-effort)
        try:
            sh(["git", "fetch", "--all", "--tags", "--prune"], cwd=d)
        except Exception:
            pass
        return d
    sh(["git", "clone", "--no-tags", "--filter=blob:none", "--recurse-submodules=no", url, str(d)])
    return d

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir)
    return int(cp.stdout.strip() or "0")

# %%
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {"repo_url": url, "dir": None, "status": "unknown", "seconds": None, "total_commits": None, "error": ""}
        try:
            d = ensure_cloned(url, CLONE_ROOT)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            ok += 1
        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)")

# %%
# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["repo_url","dir","status","seconds","total_commits","error"])
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[ok] https://github.com/connectbot/connectbot -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\connectbot__connectbot (1.64s)
[ok] https://github.com/robolectric/robolectric -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\robolectric__robolectric (7.22s)
[ok] https://github.com/opendocument-app/OpenDocument.droid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\opendocument-app__OpenDocument.droid (4.29s)
[ok] https://github.com/maxpower47/PinDroid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\maxpower47__PinDroid (2.95s)
[ok] https://github.com/Rajawali/Rajawali -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\Rajawali__Rajawali (6.46s)
[ok] https://github.com/cgeo/cgeo -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\cgeo__cgeo (24.03s)
[ok] https://github.com/OneBusAway/onebusaway-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\OneBusAway__onebus

In [ ]:
# %% [markdown]
# RQ2 — Step 1: Mine commit snapshots (simplified invocation feature)
# Scans cloned repos for commits touching CI/YAML/Gradle/scripts and writes per-repo JSONL snapshots.
# - Windows-safe UTF-8 decoding for all git calls (fixes cp1252 UnicodeDecodeError).
# - Robust fallbacks so a wonky repo doesn't stop the whole run.

# %%
from __future__ import annotations

import os
import re
import json
import subprocess
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT      = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
CLONE_ROOT     = WORK_ROOT / "clones"
SNAPSHOT_DIR   = WORK_ROOT / "snapshots"   # per-repo JSONL output here
MAX_COMMITS_PER_REPO = 0                   # 0 = no limit

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# Optional YAML support (safe to skip if you don't need deep YAML parsing)
try:
    import yaml  # pip install pyyaml
except Exception:
    yaml = None

# -----------------------------
# Subprocess helper (Windows-safe UTF-8)
# -----------------------------
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """
    Run a command and return CompletedProcess with UTF-8 decoding and error replacement.
    Prevents UnicodeDecodeError on Windows when reading git output.
    """
    env = os.environ.copy()
    env["GIT_PAGER"] = "cat"
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        check=check,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",   # force UTF-8 decode
        errors="replace",   # never crash on odd bytes
        env=env,
    )

# -----------------------------
# Relevant file surfaces
# -----------------------------
CI_FILES = [
    ".gitlab-ci.yml",
    ".circleci/config.yml",
    "bitrise.yml",
    "azure-pipelines.yml",
]

GRADLE_FILES = [
    "build.gradle", "build.gradle.kts",
    "settings.gradle", "settings.gradle.kts",
    "gradle.properties",
    "gradle/wrapper/gradle-wrapper.properties",
]

def is_ci_file(p: str) -> bool:
    return p.startswith(".github/workflows/") or p in CI_FILES

def is_gradle_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(x) for x in (f.lower() for f in GRADLE_FILES))

def is_script_file(p: str) -> bool:
    lp = p.lower()
    return lp.endswith(".sh") or lp.endswith(".py") or ("script" in lp)

def touched_relevant(paths: List[str]) -> bool:
    for p in paths:
        if not p.strip():
            continue
        if is_ci_file(p) or is_gradle_file(p) or is_script_file(p):
            return True
    return False

# -----------------------------
# Git helpers
# -----------------------------
def list_relevant_commits(repo_dir: Path) -> List[Tuple[str, int, List[str]]]:
    """
    Returns list of (sha, unix_ts, [changed_paths]) for commits that touched relevant files.
    Oldest -> newest order.
    """
    cp = sh([
        "git", "-c", "i18n.logOutputEncoding=UTF-8", "-c", "core.quotepath=off",
        "log", "--all", "--name-only", "--pretty=%H%x09%ct"
    ], cwd=repo_dir)
    results: List[Tuple[str, int, List[str]]] = []
    sha: Optional[str] = None
    ts: Optional[int] = None
    changed: List[str] = []
    for line in cp.stdout.splitlines():
        if re.match(r"^[0-9a-f]{40}\t\d+$", line):
            if sha is not None and touched_relevant(changed):
                results.append((sha, ts, changed))
            sha, ts_s = line.split("\t", 1)
            ts = int(ts_s)
            changed = []
        else:
            if line.strip():
                changed.append(line.strip())
    if sha is not None and touched_relevant(changed):
        results.append((sha, ts, changed))
    results.reverse()  # oldest -> newest
    return results

def git_show(repo_dir: Path, sha: str, path: str) -> Optional[str]:
    try:
        cp = sh(["git", "show", f"{sha}:{path}"], cwd=repo_dir)
        return cp.stdout
    except subprocess.CalledProcessError:
        return None

def git_subject(repo_dir: Path, sha: str) -> str:
    try:
        cp = sh(["git", "-c", "i18n.logOutputEncoding=UTF-8", "show", "-s", "--format=%s", sha],
                cwd=repo_dir, check=False)
        return (cp.stdout or "").strip()
    except Exception:
        return ""

# -----------------------------
# Heuristic extractors
# -----------------------------
RE_INT      = re.compile(r"\d+")
RE_VERSION  = re.compile(r"\b(\d+(?:\.\d+){0,3})\b")

YAML_KEYS = {
    "api": ["api-level","apilevel","api_level"],
    "abi": ["abi","arch","cpu","abi_filters","abi-filter"],
    "system_image": ["system-image","target","systemimage"],
    "device": ["device","avd-name","avd","device-profile","model","hardwareProfile"],
    "orchestrator": ["orchestrator","android-test-orchestrator","use-orchestrator"],
    "wait": ["wait-for-boot","wait_for_boot"],
    "timeouts": ["emulator-boot-timeout","timeout","test-timeout","emulator_timeout"],
    "retries": ["retry","retries","max-retries"],
    "matrix": ["matrix","strategy"],
    "runner_os": ["runs-on","machine","image"],
    "jdk": ["java-version","jdk","java","distribution"],
    "invocation": ["run","gradle_args","gradlew_args","task","tasks"],
    "thirdparty": ["browserstack","saucelabs","firebase","bitbar","kobiton","testlab","devicefarm"],
}

# --- Simplified invocation classification (2D: style + tags) ---

DIY_RX = re.compile(
    r"(?:\bemulator(?:\.bat)?\s-|"
    r"\bavdmanager\b|"
    r"\bsdkmanager\b|"
    r"\bcreate\s+avd\b|"
    r"\badb\s+(?:-s\s+\S+\s+)?wait-for-device\b|"
    r"\badb\s+shell\s+getprop\s+sys\.boot_completed\b|"
    r"\bqemu\b)",
    re.I,
)

GMD_RX = re.compile(
    r"(?:\bmanaged\s+device\b|\bgradle\s+managed\s+device\b|\bPixel\w*Api\d+\b)",
    re.I,
)

GMD_GRADLE_RX = re.compile(
    r"(?:\bmanagedDevices\s*\{|testOptions\s*\{[^}]*devices)",
    re.I | re.S,
)

CONNECTED_RX = re.compile(r"\bconnectedAndroidTest\b", re.I)

TAG_PATTERNS = {
    "headless":   re.compile(r"-no-window|-headless", re.I),
    "gpu":        re.compile(r"-gpu\s+\w+", re.I),
    "no_window":  re.compile(r"-no-window|-headless", re.I),
    "concurrency":re.compile(r"--parallel|\bmax-workers\b|\borg\.gradle\.workers\.max\b", re.I),
    "sharding":   re.compile(r"\bnumShards\b|\bshard(?:ing)?\b", re.I),
    "instr_args": re.compile(r"-Pandroid\.testInstrumentationRunnerArguments\.", re.I),
}

def classify_invocation_from_texts(yaml_snippets: List[str], gradle_texts: List[str]) -> tuple[str, List[str]]:
    """
    Returns (invocation_style, invocation_tags)
    - yaml_snippets: values from 'run'/'task'/'args' collected from YAML
    - gradle_texts: full text of any Gradle files in the same snapshot (optional)
    """
    hay = "\n".join(s for s in yaml_snippets if s)
    gmd_hit = bool(GMD_RX.search(hay)) or any(GMD_GRADLE_RX.search(t or "") for t in gradle_texts)
    diy_hit = bool(DIY_RX.search(hay))
    connected_hit = bool(CONNECTED_RX.search(hay))

    if gmd_hit:
        style = "gmd"
    elif diy_hit:
        style = "diy"
    elif connected_hit:
        style = "gradle_connected"
    else:
        style = "unknown"

    tags = sorted({name for name, rx in TAG_PATTERNS.items() if rx.search(hay)})
    return style, tags

# -----------------------------
# YAML extractor
# -----------------------------
def extract_from_yaml_text(text: str) -> Dict[str, Any]:
    if yaml is None:
        return {}
    try:
        docs = list(yaml.safe_load_all(text))
    except Exception:
        docs = []
    out = {
        # real features
        "api_levels": set(), "abis": set(), "system_images": set(), "device_profiles": set(),
        "orchestrator": None, "wait_for_boot": None, "timeouts": {}, "retries": None,
        "matrix_axes": set(), "runner_os": None, "jdk": None,
        # study-defined (simplified)
        "invocation_style": "unknown", "invocation_tags": set(),
        "thirdparty_refs": set(),
    }
    # temp bucket: raw run/task/args strings for classification
    _invocation_snippets: List[str] = []

    def scan_obj(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                lk = str(k).lower()
                if lk in (x.lower() for x in YAML_KEYS["api"]):
                    if isinstance(v, list):
                        for x in v:
                            if isinstance(x, (int,str)) and RE_INT.search(str(x)):
                                out["api_levels"].add(int(RE_INT.search(str(x)).group()))
                    elif isinstance(v, (int,str)):
                        m = RE_INT.search(str(v))
                        if m: out["api_levels"].add(int(m.group()))
                if lk in (x.lower() for x in YAML_KEYS["abi"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["abis"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["system_image"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["system_images"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["device"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): out["device_profiles"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["orchestrator"]):
                    if isinstance(v, bool): out["orchestrator"] = v
                    elif isinstance(v, str): out["orchestrator"] = v.lower() in ("1","true","yes","on")
                if lk in (x.lower() for x in YAML_KEYS["wait"]):
                    if isinstance(v, bool): out["wait_for_boot"] = v
                    elif isinstance(v, str): out["wait_for_boot"] = v.lower() in ("1","true","yes","on")
                if lk in (x.lower() for x in YAML_KEYS["timeouts"]):
                    out["timeouts"][k] = v
                if lk in (x.lower() for x in YAML_KEYS["retries"]):
                    try: out["retries"] = int(RE_INT.search(str(v)).group())
                    except Exception: pass
                if lk in (x.lower() for x in YAML_KEYS["matrix"]):
                    if isinstance(v, dict):
                        for ax, _vals in v.items():
                            out["matrix_axes"].add(str(ax))
                if lk in (x.lower() for x in YAML_KEYS["runner_os"]):
                    out["runner_os"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["jdk"]):
                    out["jdk"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["invocation"]):
                    _invocation_snippets.append(str(v))
                if lk in (x.lower() for x in YAML_KEYS["thirdparty"]):
                    out["thirdparty_refs"].add(lk)
                if isinstance(v, (dict, list)):
                    scan_obj(v)
        elif isinstance(obj, list):
            for x in obj:
                scan_obj(x)

    for d in docs:
        scan_obj(d)

    # classify invocation from YAML-only snippets
    style, tags = classify_invocation_from_texts(_invocation_snippets, [])
    out["invocation_style"] = style
    out["invocation_tags"].update(tags)

    # convert sets to sorted lists for JSON
    out["api_levels"]      = sorted(out["api_levels"])
    out["abis"]            = sorted(out["abis"])
    out["system_images"]   = sorted(out["system_images"])
    out["device_profiles"] = sorted(out["device_profiles"])
    out["matrix_axes"]     = sorted(out["matrix_axes"])
    out["invocation_tags"] = sorted(out["invocation_tags"])
    out["thirdparty_refs"] = sorted(out["thirdparty_refs"])
    return out

# -----------------------------
# Single-file extractor (YAML + Gradle)
# -----------------------------
def extract_from_text(path: str, text: str) -> Dict[str, Any]:
    # YAML configs
    if path.lower().endswith((".yml", ".yaml")) and yaml is not None:
        data = extract_from_yaml_text(text)
    else:
        data = {}

    # Gradle heuristics
    if path.endswith(("build.gradle", "build.gradle.kts",
                      "gradle.properties", "gradle/wrapper/gradle-wrapper.properties",
                      "settings.gradle", "settings.gradle.kts")):
        agp = re.findall(r"com\.android\.tools\.build:gradle:([0-9][^'\"\s]+)", text)
        if agp:
            data["agp_versions"] = sorted(set(agp))
        for m in re.finditer(r"apiLevel\s*=\s*(\d+)", text, re.IGNORECASE):
            data.setdefault("api_levels", [])
            lvl = int(m.group(1))
            if lvl not in data["api_levels"]:
                data["api_levels"].append(lvl)
        if re.search(r"ANDROIDX_TEST_ORCHESTRATOR", text):
            data["orchestrator"] = True

        # NEW: recognize GMD via Gradle DSL (even if YAML 'run' strings don't mention it)
        if GMD_GRADLE_RX.search(text):
            # If YAML already set a style, don't downgrade it; prefer gmd when detected
            current = data.get("invocation_style")
            if current in (None, "", "unknown"):
                data["invocation_style"] = "gmd"
            elif current == "diy":  # prefer explicit GMD DSL over DIY run strings if both appear
                data["invocation_style"] = "gmd"

    return data

# -----------------------------
# IO helpers
# -----------------------------
def write_jsonl(path: Path, rows: List[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# %%
# -----------------------------
# Main mining loop
# -----------------------------
repos = [p for p in CLONE_ROOT.iterdir() if (p / ".git").exists()]
repos.sort(key=lambda p: p.name.lower())
print(f"Found {len(repos)} repos in {CLONE_ROOT}")

ok_count = 0
skip_count = 0
err_count = 0

for repo in repos:
    try:
        rel_commits = list_relevant_commits(repo)
        if MAX_COMMITS_PER_REPO > 0:
            rel_commits = rel_commits[:MAX_COMMITS_PER_REPO]

        out_rows: List[dict] = []
        for sha, ts, changed_paths in rel_commits:
            # keep only the relevant paths from that commit
            rel_paths = [p for p in changed_paths if is_ci_file(p) or is_gradle_file(p) or is_script_file(p)]
            if not rel_paths:
                continue
            subj = git_subject(repo, sha)
            for pth in rel_paths:
                txt = git_show(repo, sha, pth)
                if txt is None:
                    continue
                feats = extract_from_text(pth, txt)
                out_rows.append({
                    "repo": repo.name,
                    "sha": sha,
                    "timestamp": ts,
                    "subject": subj,
                    "path": pth,
                    "features": feats,
                })

        if not out_rows:
            print(f"[skip] {repo.name}: no relevant snapshots")
            skip_count += 1
            continue

        dst = SNAPSHOT_DIR / f"{repo.name}.jsonl"
        write_jsonl(dst, out_rows)
        print(f"[ok] {repo.name}: {len(out_rows)} snapshots -> {dst}")
        ok_count += 1

    except Exception as e:
        # Never crash the whole batch; log and continue
        print(f"[err] {repo.name}: {e}")
        err_count += 1

print(f"\nDone. ok={ok_count}, skip={skip_count}, err={err_count}, out_dir={SNAPSHOT_DIR}")


Found 282 repos in C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones
[ok] 4eRTuk__audioview: 62 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 175 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 157 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 47 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\AAkira__ExpandableLayout.jsonl
[ok] abdelaziz-mahdy__pytorch_lite: 133 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\abdelaziz-mahdy__pytorch_lite.jsonl
[ok] ably__ably-flutter: 369 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\ably__ably-flutter.jsonl
[ok] AChep__AcDisplay: 124 snapshots -> C:\Android Mobile App

In [ ]:
# RQ2 — Step 2 (Label+) : Field-level deltas + commit/delta secondary labels + Driver (V3.0)
# - Reads Step 1 snapshots and emits enriched CCE rows (per field change).
# - Commit-side secondary labels: CSV-driven (Regex).
# - Delta-side secondary labels: deterministic, from field-level diffs.
# - Publishes per-source labels and their union; resolves one Driver per CCE using fixed priority.
# - Records timestamps explicitly as UTC (epoch + ISO 8601 Z).

from __future__ import annotations
import os, json, re, csv
import datetime as _dt
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshots"              # input from Step 1 (Mine)
CCE_ENRICHED_DIR  = WORK_ROOT / "cce_enriched_V3.0"      # output: enriched episodes (per-repo JSONL)
CCE_ENRICHED_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Detection list (CSV) location
# -----------------------------
DETECTION_LIST_DIR = Path(
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V3.0\Second_Label_Intent"
)
# Optional: allow a full file path override via env var
DETECTION_CSV_ENV = os.environ.get("DETECTION_LIST_CSV")   # full CSV path if set

# -----------------------------
# Helpers
# -----------------------------
VERSION_RE = re.compile(r"\d+(?:\.\d+)*")

# Primary label mapping (what changed)
FIELD_TO_PRIMARY = {
    "api_levels":        "api_bump",
    "agp_versions":      "agp_bump",
    "jdk":               "jdk_bump",
    "runner_os":         "runner_os_change",
    "matrix_axes":       "matrix_change",
    "orchestrator":      "orchestrator_toggle",
    "timeouts":          "timeout_tuning",
    "retries":           "retry_tuning",
    "device_profiles":   "device_profile_change",
    "abis":              "abi_change",
    "system_images":     "system_image_change",
    "invocation_hints":  "invocation_change",
    "thirdparty_refs":   "external_service_change",
    "wait_for_boot":     "wait_strategy_change",
}

def read_snapshots(folder: Path) -> Dict[str, List[dict]]:
    by_repo: Dict[str, List[dict]] = {}
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                repo = d.get("repo")
                if not repo:
                    continue
                by_repo.setdefault(repo, []).append(d)
    for repo, rows in by_repo.items():
        rows.sort(key=lambda r: (r.get("path",""), int(r.get("timestamp", 0)), r.get("sha","")))
    return by_repo

def as_set_str(xs) -> set:
    if xs is None:
        return set()
    if isinstance(xs, (list, set, tuple)):
        return set(str(x) for x in xs)
    return {str(xs)}

def as_set_int(xs) -> set:
    if xs is None:
        return set()
    out = set()
    if isinstance(xs, (list, set, tuple)):
        for x in xs:
            try:
                out.add(int(x))
            except Exception:
                pass
    else:
        try:
            out.add(int(xs))
        except Exception:
            pass
    return out

def parse_version_tuple(s: str) -> Tuple[int, ...]:
    if not s:
        return tuple()
    m = VERSION_RE.search(str(s))
    if not m:
        return tuple()
    parts = m.group(0).split(".")
    out: List[int] = []
    for p in parts:
        try:
            out.append(int(p))
        except Exception:
            out.append(0)
    return tuple(out)

def max_version_tuple(strings: List[str]) -> Tuple[int, ...]:
    best: Tuple[int, ...] = tuple()
    for s in strings or []:
        vt = parse_version_tuple(str(s))
        if vt > best:
            best = vt
    return best

def stringify(x: Any) -> str:
    if isinstance(x, (dict, list, set, tuple)):
        try:
            return json.dumps(x, ensure_ascii=False, sort_keys=True)
        except Exception:
            return str(x)
    return "" if x is None else str(x)

def _safe_json_loads(s: str):
    try:
        return json.loads(s) if s else None
    except Exception:
        return None

def diff_features(old: Dict[str, Any], new: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    Field-level changes between two feature dicts.
    Returns [{field, old_value, new_value, change_type, magnitude?, added_items?, removed_items?}, ...]
    Values are stringified JSON so they are safe to serialize; callers can _safe_json_loads().
    """
    old = old or {}
    new = new or {}
    out: List[Dict[str, Any]] = []

    def handle_set(field: str, to_set_fn):
        a = to_set_fn(old.get(field))
        b = to_set_fn(new.get(field))
        if a == b:
            return
        added   = sorted(b - a)
        removed = sorted(a - b)
        row = {
            "field": field,
            "old_value": stringify(sorted(a)),
            "new_value": stringify(sorted(b)),
            "change_type": "modified",
        }
        if added:   row["added_items"] = stringify(added)
        if removed: row["removed_items"] = stringify(removed)
        if field == "api_levels" and a and b:
            try:
                row["magnitude"] = max(b) - max(a)
            except Exception:
                pass
        out.append(row)

    handle_set("api_levels", as_set_int)
    handle_set("abis", as_set_str)
    handle_set("system_images", as_set_str)
    handle_set("device_profiles", as_set_str)
    handle_set("matrix_axes", as_set_str)
    handle_set("invocation_hints", as_set_str)
    handle_set("thirdparty_refs", as_set_str)

    # AGP versions
    a_agp = sorted(as_set_str(old.get("agp_versions")))
    b_agp = sorted(as_set_str(new.get("agp_versions")))
    if a_agp != b_agp:
        row = {
            "field": "agp_versions",
            "old_value": stringify(a_agp),
            "new_value": stringify(b_agp),
            "change_type": "modified",
        }
        old_max = max_version_tuple(list(a_agp))
        new_max = max_version_tuple(list(b_agp))
        if old_max or new_max:
            row["magnitude"] = 1 if new_max > old_max else (-1 if new_max < old_max else 0)
        out.append(row)

    # Scalars
    for field in ("orchestrator","wait_for_boot","retries","runner_os","jdk"):
        a = old.get(field, None)
        b = new.get(field, None)
        if a != b:
            out.append({
                "field": field,
                "old_value": stringify(a),
                "new_value": stringify(b),
                "change_type": (
                    "modified" if (a is not None and b is not None)
                    else ("added" if a is None else "removed")
                ),
            })

    # Dict-ish
    for field in ("timeouts",):
        a = old.get(field, None)
        b = new.get(field, None)
        if stringify(a) != stringify(b):
            out.append({
                "field": field,
                "old_value": stringify(a),
                "new_value": stringify(b),
                "change_type": (
                    "modified" if (a is not None and b is not None)
                    else ("added" if a is None else "removed")
                ),
            })

    return out

# -----------------------------
# CSV-driven commit intent patterns
# -----------------------------
@dataclass
class IntentPattern:
    label: str
    regex: re.Pattern

_DET_FILENAMES = [
    "Detection_List_V3.0.csv",
    "Detection_List.csv",
    "detection_list.csv",
]

def _resolve_csv_path() -> Path:
    # 1) env var direct file?
    if DETECTION_CSV_ENV:
        p = Path(DETECTION_CSV_ENV)
        if p.exists() and p.is_file():
            return p
    # 2) forced directory: common names then any *.csv
    if DETECTION_LIST_DIR.exists() and DETECTION_LIST_DIR.is_dir():
        for name in _DET_FILENAMES:
            p = DETECTION_LIST_DIR / name
            if p.exists():
                return p
        csvs = sorted(DETECTION_LIST_DIR.glob("*.csv"))
        if csvs:
            return csvs[0]
    # 3) fallbacks
    fallbacks = [
        WORK_ROOT / "Detection_List_V3.0.csv",
        Path.cwd() / "Detection_List_V3.0.csv",
        Path(r"/mnt/data/Detection_List_V3.0.csv"),
    ]
    for p in fallbacks:
        if p.exists():
            return p
    raise FileNotFoundError(
        "Detection list CSV not found.\n"
        f"Tried env DETECTION_LIST_CSV, directory: {DETECTION_LIST_DIR}, and fallbacks:\n  "
        + "\n  ".join(str(x) for x in fallbacks)
    )

def _detect_headers(header: List[str]) -> Tuple[str, str]:
    lowered = {h.lower(): h for h in header}
    for cand in ("intent_label", "label", "intent", "group", "intentname"):
        if cand in lowered:
            label_col = lowered[cand]
            break
    else:
        raise KeyError("Could not find an intent label column in CSV headers: " + ", ".join(header))
    for cand in ("regex", "pattern", "regexp", "re"):
        if cand in lowered:
            regex_col = lowered[cand]
            break
    else:
        if "Regex" in header:
            regex_col = "Regex"
        else:
            raise KeyError("Could not find a regex column in CSV headers: " + ", ".join(header))
    return label_col, regex_col

def _load_intent_patterns() -> List[IntentPattern]:
    csv_path = _resolve_csv_path()
    intents: List[IntentPattern] = []
    with csv_path.open(encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        if reader.fieldnames is None:
            raise RuntimeError(f"{csv_path} has no header row.")
        label_col, regex_col = _detect_headers(reader.fieldnames)
        for i, row in enumerate(reader, start=2):
            label = (row.get(label_col) or "").strip()
            pattern = (row.get(regex_col) or "").strip()
            if not label or not pattern:
                continue
            try:
                rx = re.compile(pattern, re.MULTILINE | re.IGNORECASE)
            except re.error as e:
                print(f"[warn] bad regex at line {i} for label '{label}': {e}")
                continue
            intents.append(IntentPattern(label=label, regex=rx))
    if not intents:
        raise RuntimeError(f"No valid intent patterns loaded from {csv_path}")
    print(f"[ok] loaded {len(intents)} intent patterns")
    return intents

INTENT_PATTERNS: List[IntentPattern] = _load_intent_patterns()

# Prefer 'revert' over upgrade-like labels when both match
UPGRADE_LIKE = {"update/upgrade", "upgrade"}

# -----------------------------
# Intent taxonomy & driver mapping
# -----------------------------
AUX_LABELS = {
    "revert", "cleanup_refactor", "removal_deprecation",
    "docs_changelog", "chore", "release_tagging"
}  # recorded, never used to choose driver

LABEL_TO_CATEGORY = {
    # commit-side → driver categories
    "deflake": "stability_tuning",
    "fail_fix": "stability_tuning",
    "speed_up": "stability_tuning",
    "ci_migration": "ci_env_workflow",
    "ci_config_change": "ci_env_workflow",
    "update_upgrade": "version_change",
    "version_change": "version_change",

    # delta-side → driver categories
    "expand_coverage": "coverage",
    "reduce_coverage": "coverage",
    "coverage_dimensions_change": "coverage",
    "timeout_tuning": "stability_tuning",
    "retry_tuning": "stability_tuning",
    "orchestrator_change": "stability_tuning",
    "flake_mitigation": "stability_tuning",
    "speed_up_ci": "stability_tuning",
    "infra_tuning": "ci_env_workflow",
    "external_service_change": "ci_env_workflow",
    "adopt_gmd": "version_change",
    "drop_gmd": "version_change",
    "adopt_diy": "version_change",
}

DRIVER_PRIORITY = ["stability_tuning", "coverage", "ci_env_workflow", "version_change"]

# -----------------------------
# Commit-side secondary labels
# -----------------------------
def derive_commit_labels(subject: str) -> Tuple[List[str], List[str]]:
    """
    Returns (commit_labels, aux_tags).
    commit_labels includes ALL matched labels (incl. aux),
    but we also return aux_tags so the driver resolver can ignore them.
    """
    s = subject or ""
    labels: set[str] = set()
    for ip in INTENT_PATTERNS:
        try:
            if ip.regex.search(s):
                labels.add(ip.label)
        except Exception:
            continue
    # Prefer revert over upgrade-like
    if "revert" in labels and any(lbl in labels for lbl in UPGRADE_LIKE):
        labels -= UPGRADE_LIKE
    aux = sorted(lbl for lbl in labels if lbl in AUX_LABELS)
    non_aux = sorted(lbl for lbl in labels if lbl not in AUX_LABELS)
    return non_aux + aux, aux

# -----------------------------
# Delta-side secondary labels
# -----------------------------
_DURATION_RX = re.compile(r"(?i)^\s*(\d+(?:\.\d+)?)\s*(ms|s|m|h)?\s*$")
_UNIT_TO_SEC = {"ms": 0.001, "s": 1, "m": 60, "h": 3600}

def _to_seconds(x) -> Optional[float]:
    if x is None:
        return None
    if isinstance(x, (int, float)):
        return float(x)
    m = _DURATION_RX.match(str(x))
    if not m:
        return None
    val = float(m.group(1))
    unit = (m.group(2) or "s").lower()
    return val * _UNIT_TO_SEC.get(unit, 1)

def _sum_timeout_seconds(obj) -> Optional[float]:
    """
    Accept dict-like {'setup': '30s', 'test':'90s'} or list of pairs.
    Returns total seconds if at least one value parsed; else None.
    """
    if obj is None:
        return None
    total = 0.0
    seen = 0
    if isinstance(obj, dict):
        it = obj.values()
    elif isinstance(obj, list):
        it = [v for _, v in obj if isinstance(_, (str,int))]
    else:
        return None
    for v in it:
        secs = _to_seconds(v)
        if secs is not None:
            total += secs
            seen += 1
    return total if seen else None

def derive_delta_labels(deltas: List[Dict[str, Any]]) -> List[str]:
    labs: set[str] = set()
    for d in deltas:
        field = d.get("field")
        old_v = d.get("old_value")
        new_v = d.get("new_value")
        added = _safe_json_loads(d.get("added_items") or "") or []
        removed = _safe_json_loads(d.get("removed_items") or "") or []

        # Coverage sets
        if field in {"api_levels", "abis", "device_profiles", "system_images"}:
            if added:   labs.add("expand_coverage")
            if removed: labs.add("reduce_coverage")

        # Matrix axes → coverage dimensions + expand/reduce
        if field == "matrix_axes":
            labs.add("coverage_dimensions_change")
            if added:   labs.add("expand_coverage")
            if removed: labs.add("reduce_coverage")

        # Timeouts
        if field == "timeouts":
            labs.add("timeout_tuning")
            old_obj = _safe_json_loads(old_v)
            new_obj = _safe_json_loads(new_v)
            old_s = _sum_timeout_seconds(old_obj)
            new_s = _sum_timeout_seconds(new_obj)
            if old_s is not None and new_s is not None:
                if new_s > old_s:
                    labs.add("flake_mitigation")
                elif new_s < old_s:
                    labs.add("speed_up_ci")

        # Retries
        if field == "retries":
            labs.add("retry_tuning")
            try:
                a = int(_safe_json_loads(old_v) if old_v else 0)
            except Exception:
                a = None
            try:
                b = int(_safe_json_loads(new_v) if new_v else 0)
            except Exception:
                b = None
            if a is not None and b is not None:
                if b > a:
                    labs.add("flake_mitigation")
                elif b < a:
                    labs.add("speed_up_ci")

        # Orchestrator
        if field == "orchestrator":
            labs.add("orchestrator_change")
            # Turning it on generally mitigates flakiness; we don't try to read polarity here.

        # Runner platform
        if field == "runner_os":
            labs.add("infra_tuning")

        # Invocation strategy
        if field == "invocation_hints":
            if "gmd" in added:   labs.add("adopt_gmd")
            if "gmd" in removed: labs.add("drop_gmd")
            if "diy" in added:   labs.add("adopt_diy")

        # External device-cloud integrations
        if field == "thirdparty_refs":
            labs.add("external_service_change")

    return sorted(labs)

# -----------------------------
# Driver resolver
# -----------------------------
def resolve_driver(secondary_labels_union: List[str], aux_tags: List[str]) -> Optional[str]:
    # Remove aux labels from consideration
    effective = [lbl for lbl in secondary_labels_union if lbl not in AUX_LABELS]
    # Map labels to categories
    cats: set[str] = set()
    for lbl in effective:
        cat = LABEL_TO_CATEGORY.get(lbl)
        if cat:
            cats.add(cat)
    # Choose by fixed priority
    for cat in DRIVER_PRIORITY:
        if cat in cats:
            return cat
    return None  # no secondary labels → driver remains null

# -----------------------------
# UTC timestamp helpers
# -----------------------------
def ensure_utc_epoch(ts_any) -> int:
    """
    Accepts int/str/float and returns integer epoch seconds in UTC.
    Step 1 uses `git log %ct` which is already epoch seconds; we normalize anyway.
    """
    try:
        ts = int(float(ts_any))
    except Exception:
        ts = 0
    return ts

def epoch_to_iso_utc(ts: int) -> str:
    return _dt.datetime.fromtimestamp(int(ts), tz=_dt.timezone.utc).isoformat().replace("+00:00", "Z")

# -----------------------------
# Main: build enriched episodes
# -----------------------------
if __name__ == "__main__":
    by_repo = read_snapshots(SNAPSHOT_DIR)
    print(f"Loaded snapshots for {len(by_repo)} repos from {SNAPSHOT_DIR}")

    for repo, rows in by_repo.items():
        out_rows: List[dict] = []
        by_path: Dict[str, List[dict]] = {}
        for r in rows:
            by_path.setdefault(r.get("path",""), []).append(r)

        # Track repeats per (path, field) within the repo
        repeat_counter: Dict[Tuple[str,str], int] = {}

        for path, snaps in by_path.items():
            prev: Optional[dict] = None
            for cur in snaps:
                if prev is None:
                    prev = cur
                    continue

                old_feats = prev.get("features", {}) or {}
                new_feats = cur.get("features", {}) or {}
                deltas = diff_features(old_feats, new_feats)

                if deltas:
                    # --- Episode-level (prev -> cur) labels ---
                    commit_labels_all, aux_tags = derive_commit_labels(cur.get("subject",""))
                    delta_labels_all = derive_delta_labels(deltas)
                    secondary_union = sorted(set(commit_labels_all) | set(delta_labels_all))
                    driver = resolve_driver(secondary_union, aux_tags)

                    # Timestamp: force UTC
                    ts_epoch_utc = ensure_utc_epoch(cur.get("timestamp", 0))
                    ts_iso_utc   = epoch_to_iso_utc(ts_epoch_utc)

                    for d in deltas:
                        key = (path, d["field"])
                        repeat_counter[key] = repeat_counter.get(key, 0) + 1
                        repeat_index = repeat_counter[key]
                        repeat_label = "first" if repeat_index == 1 else "repeat"

                        primary_label = FIELD_TO_PRIMARY.get(d["field"], "other_change")

                        out_rows.append({
                            "repo": repo,
                            "sha": cur.get("sha"),
                            "prev_sha": prev.get("sha"),

                            # UTC timestamps (both forms)
                            "timestamp_epoch_utc": ts_epoch_utc,  # integer seconds since epoch (UTC)
                            "timestamp_utc": ts_iso_utc,          # ISO-8601 'Z' (UTC)
                            # (For backward compatibility; you can drop this later if you want)
                            "timestamp": ts_epoch_utc,

                            "path": path,
                            "subject": cur.get("subject",""),

                            # Field-level delta
                            "field": d["field"],
                            "old_value": d.get("old_value",""),
                            "new_value": d.get("new_value",""),
                            "change_type": d.get("change_type","modified"),
                            "magnitude": d.get("magnitude", None),
                            "added_items": d.get("added_items",""),
                            "removed_items": d.get("removed_items",""),

                            "repeat_index": repeat_index,
                            "repeat_label": repeat_label,

                            # Labels
                            "primary_label": primary_label,
                            "secondary_labels_commit": sorted(commit_labels_all),
                            "secondary_labels_delta": sorted(delta_labels_all),
                            "secondary_labels": secondary_union,
                            "aux_tags": aux_tags,

                            # Single Driver per CCE (from union, aux ignored)
                            "driver": driver,  # one of: stability_tuning, coverage, ci_env_workflow, version_change, or null
                        })
                prev = cur

        if not out_rows:
            print(f"[skip] {repo}: no field-level deltas found")
            continue

        dst = CCE_ENRICHED_DIR / f"{repo}.jsonl"
        with dst.open("w", encoding="utf-8") as f:
            for r in out_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[ok] {repo}: {len(out_rows)} enriched rows -> {dst}")

    print("Done.")


[warn] bad regex at line 7 for label 'speed_up': unbalanced parenthesis at position 186
[ok] loaded 13 intent patterns from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Second_Label\Method_V3.0\Second_Label_Intent\Detection_List_V3.0.csv
Loaded snapshots for 282 repos from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots
[ok] 4eRTuk__audioview: 8 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.0\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 52 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.0\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 16 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.0\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 3 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3.0\AAkira__ExpandableLayout.jsonl
[ok] abdelaziz-mahdy__pytorch_lite: 3

In [ ]:
# %% [markdown]
# # RQ2 — Step 3 (Combine+): Snapshots + Enriched Episodes (+ change_op fields)
# Combines per-repo JSONL into CSV (and Parquet if pandas/pyarrow available).
# - Snapshots: keep `features_json` as a string (drop raw `features`)
# - Episodes (from Step 2): normalize label columns from the new schema and add change classification:
#     * `secondary_labels_str`, `secondary_labels_json`, `secondary_label_count`, `has_secondary_label`
#     * `secondary_labels_commit_str/json/count/has_*`
#     * `secondary_labels_delta_str/json/count/has_*`
#     * `aux_tags_str/json/aux_tag_count/has_aux_tag`
#     * `driver` (as produced in Step 2) + `has_driver`
#     * `change_op` in {"add","remove","value_edit","no_change"}
#     * `is_major_emulator_change` (1 if add/remove else 0)
# - Timestamps: respects `timestamp_epoch_utc` and `timestamp_utc` (keeps legacy `timestamp`)
# - Parquet: enforce predictable dtypes

from __future__ import annotations
import json, csv, os, re, ast, math
from pathlib import Path
from typing import List, Any, Optional, Tuple

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshots"             # Step 1 output
CCE_ENRICHED_DIR  = WORK_ROOT / "cce_enriched_V3.0"     # Step 2 output (new schema)
COMBINE_DIR       = WORK_ROOT / "combined_V3.0"

COMBINE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers (existing)
# -----------------------------
def read_all_jsonl(folder: Path) -> List[dict]:
    out: List[dict] = []
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    d = json.loads(line)
                    d["_source_file"] = p.name
                    out.append(d)
    return out

def to_json(x: Any) -> str:
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except Exception:
        return "" if x is None else str(x)

def to_list(x: Any) -> list:
    """Tolerant listifier for list/str/json-str/None."""
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, (set, tuple)):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        # try JSON list
        if s.startswith("[") and s.endswith("]"):
            try:
                v = json.loads(s)
                return v if isinstance(v, list) else [s]
            except Exception:
                pass
        # try semicolon-separated
        if ";" in s:
            parts = [t.strip() for t in s.split(";") if t.strip()]
            return parts
        return [s] if s else []
    return [x]

def normalize_listlike(val: Any) -> tuple[str, str, int, bool]:
    """Return (str_joined, json_str, count, has_any). De-dup + sort."""
    items = [str(t).strip() for t in to_list(val) if str(t).strip()]
    items = sorted(set(items))
    s = ";".join(items)
    j = to_json(items)
    return s, j, len(items), bool(items)

# -----------------------------
# New: change classification helpers (unchanged)
# -----------------------------
CANDIDATE_COL_PAIRS = [
    ("old_value", "new_value"),
    ("value_old", "value_new"),
    ("prev_value", "curr_value"),
    ("previous_value", "current_value"),
    ("before", "after"),
    ("value_before", "value_after"),
    ("lhs", "rhs"),
    ("from_value", "to_value"),
    ("from", "to"),
    ("baseline_value", "value"),
]

def _is_na_like(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, float):
        try:
            return math.isnan(x)
        except Exception:
            return False
    if isinstance(x, str):
        s = x.strip().lower()
        return s in {"", "null", "none", "nan", "na"}
    return False

def to_none_if_empty(x: Any):
    return None if _is_na_like(x) else x

def normalize_scalar(s: str):
    if not isinstance(s, str):
        return s
    s2 = s.strip()
    if s2 == "":
        return ""
    try:
        return float(s2)  # numeric normalization
    except Exception:
        pass
    # collapse intra-string whitespace
    return re.sub(r"\s+", " ", s2)

def try_literal_or_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        pass
    try:
        return ast.literal_eval(s)  # e.g., "['arm64','x86_64']"
    except Exception:
        return s

def make_json_safe(obj):
    if isinstance(obj, float):
        return obj
    if isinstance(obj, (list, tuple)):
        return [make_json_safe(x) for x in obj]
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    return obj

def canonicalize(v):
    v = to_none_if_empty(v)
    if v is None:
        return None
    v_parsed = try_literal_or_json(v) if isinstance(v, str) else v
    if isinstance(v_parsed, dict):
        return json.dumps(make_json_safe(v_parsed), sort_keys=True, ensure_ascii=False)
    if isinstance(v_parsed, (list, tuple, set)):
        seq = list(v_parsed)
        normalized = [normalize_scalar(str(x)) if not isinstance(x, (dict, list, tuple, set)) else x for x in seq]
        try:
            return json.dumps(sorted(make_json_safe(normalized)), ensure_ascii=False)
        except Exception:
            return json.dumps(make_json_safe(normalized), ensure_ascii=False)
    if isinstance(v_parsed, str):
        return normalize_scalar(v_parsed)
    return v_parsed

def classify_change(old, new):
    old_c, new_c = canonicalize(old), canonicalize(new)
    if old_c is None and new_c is None:
        return "no_change"
    if old_c is None and new_c is not None:
        return "add"
    if old_c is not None and new_c is None:
        return "remove"
    if old_c != new_c:
        return "value_edit"
    return "no_change"

def find_value_columns_from_records(records: List[dict]) -> Tuple[Optional[str], Optional[str]]:
    """
    Auto-detect the old/new value columns from the union of keys in `records`.
    Returns (left_col_name, right_col_name) in original casing or (None, None).
    """
    # union of keys (original casing)
    all_keys: List[str] = []
    for r in records:
        for k in r.keys():
            if k not in all_keys:
                all_keys.append(k)

    lower_to_orig = {}
    for k in all_keys:
        kl = k.lower()
        if kl not in lower_to_orig:
            lower_to_orig[kl] = k

    keys_lower = set(lower_to_orig.keys())

    # exact pair match
    for left, right in CANDIDATE_COL_PAIRS:
        if left.lower() in keys_lower and right.lower() in keys_lower:
            return lower_to_orig[left.lower()], lower_to_orig[right.lower()]

    # regex fallback (heuristic)
    left_candidates  = [k for k in all_keys if re.search(r"(old|prev|before|baseline)", k, re.I)]
    right_candidates = [k for k in all_keys if re.search(r"(new|curr|after|current|to\b|value$)", k, re.I)]
    if left_candidates and right_candidates:
        return left_candidates[0], right_candidates[0]

    return None, None

# -----------------------------
# Load inputs
# -----------------------------
snapshots = read_all_jsonl(SNAPSHOT_DIR)
episodes  = read_all_jsonl(CCE_ENRICHED_DIR)
print(f"Loaded {len(snapshots)} snapshots; {len(episodes)} enriched episode rows.")

# -----------------------------
# Write CSV (snapshots) — flatten features to JSON string
# -----------------------------
snap_csv = COMBINE_DIR / "snapshots_combined.csv"
if snapshots:
    snaps_flat = []
    for r in snapshots:
        feats = r.get("features", {})
        rr = {**r, "features_json": to_json(feats)}
        rr.pop("features", None)  # drop raw features to avoid CSV/Parquet issues
        snaps_flat.append(rr)

    keys = sorted(set().union(*[set(x.keys()) for x in snaps_flat]))
    with snap_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in snaps_flat:
            w.writerow(r)
    print(f"[ok] {snap_csv}")
else:
    print("[warn] No snapshots found.")

# -----------------------------
# Write CSV (episodes enriched) — normalize label columns + add change_op fields
# -----------------------------
cce_csv = COMBINE_DIR / "episodes_enriched_combined.csv"
if episodes:
    # Detect old/new value columns once from all episode records
    left_col, right_col = find_value_columns_from_records(episodes)
    if left_col and right_col:
        print(f"[detect] value columns: old='{left_col}', new='{right_col}'")
    else:
        print("[detect] No obvious old/new value columns found; 'change_op' will be 'no_change' by default.")

    episodes_flat = []
    for r in episodes:
        # Normalized secondary label sets (union, commit-only, delta-only)
        sec_union_str, sec_union_json, sec_union_cnt, sec_union_has = normalize_listlike(r.get("secondary_labels"))
        sec_commit_str, sec_commit_json, sec_commit_cnt, sec_commit_has = normalize_listlike(r.get("secondary_labels_commit"))
        sec_delta_str,  sec_delta_json,  sec_delta_cnt,  sec_delta_has  = normalize_listlike(r.get("secondary_labels_delta"))
        aux_str, aux_json, aux_cnt, aux_has = normalize_listlike(r.get("aux_tags"))

        # Primary label
        primary = r.get("primary_label", "unknown")

        # Change op classification
        old_val = r.get(left_col) if left_col else None
        new_val = r.get(right_col) if right_col else None
        change_op = classify_change(old_val, new_val) if (left_col and right_col) else "no_change"
        is_major = int(change_op in ("add", "remove"))

        # Driver presence
        driver = r.get("driver", None)
        has_driver = int(bool(driver and str(driver).strip()))

        rr = {
            **r,

            # Keep Step-2 driver and timestamps as-is
            "driver": driver,
            "has_driver": has_driver,

            # Union labels (main)
            "secondary_labels": sec_union_str,          # compact
            "secondary_labels_str": sec_union_str,      # explicit alias
            "secondary_labels_json": sec_union_json,    # JSON list
            "secondary_label_count": sec_union_cnt,
            "has_secondary_label": int(sec_union_has),

            # Commit-only labels
            "secondary_labels_commit_str": sec_commit_str,
            "secondary_labels_commit_json": sec_commit_json,
            "secondary_label_commit_count": sec_commit_cnt,
            "has_secondary_label_commit": int(sec_commit_has),

            # Delta-only labels
            "secondary_labels_delta_str": sec_delta_str,
            "secondary_labels_delta_json": sec_delta_json,
            "secondary_label_delta_count": sec_delta_cnt,
            "has_secondary_label_delta": int(sec_delta_has),

            # Aux tags (non-driver)
            "aux_tags_str": aux_str,
            "aux_tags_json": aux_json,
            "aux_tag_count": aux_cnt,
            "has_aux_tag": int(aux_has),

            # Primary (what changed)
            "primary_label": primary,

            # Change classification
            "change_op": change_op,
            "is_major_emulator_change": is_major,
        }
        episodes_flat.append(rr)

    keys = sorted(set().union(*[set(x.keys()) for x in episodes_flat]))
    with cce_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in episodes_flat:
            w.writerow(r)
    print(f"[ok] {cce_csv}")
else:
    print("[warn] No enriched episodes found.")

# -----------------------------
# Optional: Parquet with pandas (safe dtypes)
# -----------------------------
try:
    import pandas as pd

    # Snapshots → Parquet
    if snap_csv.exists():
        df_s = pd.read_csv(snap_csv, dtype=str, keep_default_na=False)

        # Epoch timestamps if present
        for col in ("timestamp", "timestamp_epoch_utc"):
            if col in df_s.columns:
                df_s[col] = pd.to_numeric(df_s[col], errors="coerce").astype("Int64")

        df_s.to_parquet(COMBINE_DIR / "snapshots_combined.parquet", index=False)
        print("[ok] Parquet: snapshots_combined.parquet")

    # Enriched episodes → Parquet
    if cce_csv.exists():
        df_e = pd.read_csv(cce_csv, dtype=str, keep_default_na=False)

        # Ensure string dtype for mixed columns
        str_cols = [
            "old_value","new_value","added_items","removed_items","subject","path",
            "driver","field","change_type","_source_file","repo","sha","prev_sha",
            "repeat_label","primary_label","timestamp_utc",
            "secondary_labels","secondary_labels_str","secondary_labels_json",
            "secondary_labels_commit_str","secondary_labels_commit_json",
            "secondary_labels_delta_str","secondary_labels_delta_json",
            "aux_tags_str","aux_tags_json","change_op",
        ]
        for col in str_cols:
            if col in df_e.columns:
                df_e[col] = df_e[col].astype("string")

        # Numeric-friendly columns (coerce)
        for num_col in (
            "timestamp","timestamp_epoch_utc","magnitude","repeat_index",
            "secondary_label_count","secondary_label_commit_count","secondary_label_delta_count",
            "aux_tag_count","has_secondary_label","has_secondary_label_commit","has_secondary_label_delta",
            "has_aux_tag","has_driver","is_major_emulator_change"
        ):
            if num_col in df_e.columns:
                df_e[num_col] = pd.to_numeric(df_e[num_col], errors="coerce")

        # Nullable integers for epochs/indices/flags
        for i_col in ("timestamp","timestamp_epoch_utc","repeat_index",
                      "secondary_label_count","secondary_label_commit_count","secondary_label_delta_count",
                      "aux_tag_count","has_secondary_label","has_secondary_label_commit","has_secondary_label_delta",
                      "has_aux_tag","has_driver","is_major_emulator_change"):
            if i_col in df_e.columns:
                df_e[i_col] = df_e[i_col].astype("Int64")

        df_e.to_parquet(COMBINE_DIR / "episodes_enriched_combined.parquet", index=False)
        print("[ok] Parquet: episodes_enriched_combined.parquet")

except Exception as e:
    print(f"[note] Skipping Parquet (pandas/pyarrow issue?): {e}")


Loaded 224454 snapshots; 19361 enriched episode rows.
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0\snapshots_combined.csv
[detect] value columns: old='old_value', new='new_value'
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0\episodes_enriched_combined.csv
[ok] Parquet: snapshots_combined.parquet
[ok] Parquet: episodes_enriched_combined.parquet


In [6]:
"""
Normalize 'timestamp' in episodes_enriched_combined.csv

- Supports epoch as digits (sec/ms/µs/ns) and many date-like strings
- Adds:
    * timestamp_unix_s (nullable Int64)
    * timestamp_utc    (ISO 8601, Z)
    * timestamp_local  (ISO 8601 with TZ offset, America/Toronto)
- Does not overwrite the original 'timestamp' column
"""

import pandas as pd
from datetime import timezone
import numpy as np
import os

TIMEZONE = "America/Toronto"  # change if you prefer another local zone

# ---------- timezone helper (ZoneInfo -> pytz fallback) ----------
try:
    from zoneinfo import ZoneInfo  # Python 3.9+
    def _get_tz(name: str):
        return ZoneInfo(name)
except Exception:
    import pytz  # type: ignore
    def _get_tz(name: str):
        return pytz.timezone(name)

LOCAL_TZ = _get_tz(TIMEZONE)

# ---------- parsing helpers ----------
def _parse_epoch_like(s: str) -> pd.Timestamp | pd.NaT:
    """Parse digit-only epoch in sec/ms/µs/ns to UTC-aware Timestamp."""
    if s is None or (isinstance(s, float) and pd.isna(s)): 
        return pd.NaT
    s = str(s).strip()
    if not s.isdigit():
        return pd.NaT
    n = len(s)
    try:
        val = int(s)
    except Exception:
        return pd.NaT
    # infer unit from length
    if n <= 10:         # seconds
        seconds = val
    elif 11 <= n <= 15: # milliseconds
        seconds = val / 1_000
    elif 16 <= n <= 18: # microseconds
        seconds = val / 1_000_000
    else:               # nanoseconds (or larger)
        seconds = val / 1_000_000_000
    try:
        return pd.to_datetime(seconds, unit="s", utc=True)
    except Exception:
        return pd.NaT

def _parse_any_timestamp(x) -> pd.Timestamp | pd.NaT:
    """Parse any timestamp to UTC-aware Timestamp."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return pd.NaT
    s = str(x).strip()
    if s == "":
        return pd.NaT
    if s.isdigit():
        return _parse_epoch_like(s)
    # try pandas datetime (handles many formats, assumes UTC if tz-naive)
    ts = pd.to_datetime(s, utc=True, errors="coerce")
    return ts if not pd.isna(ts) else pd.NaT

def fix_timestamps(input_path: str, output_path: str | None = None, local_tz=LOCAL_TZ) -> str:
    df = pd.read_csv(input_path, dtype=str, keep_default_na=False)
    if "timestamp" not in df.columns:
        raise RuntimeError("No 'timestamp' column found in the CSV.")

    # parse to UTC
    ts_utc = df["timestamp"].apply(_parse_any_timestamp)

    # unix seconds (nullable)
    unix_s = ts_utc.view("int64") // 10**9
    unix_s = unix_s.where(~ts_utc.isna(), other=pd.NA)

    # local timezone
    ts_local = ts_utc.dt.tz_convert(local_tz)

    # ISO strings
    iso_utc   = ts_utc.dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    iso_local = ts_local.dt.strftime("%Y-%m-%dT%H:%M:%S%z")

    # append columns
    df["timestamp_unix_s"] = unix_s.astype("Int64")
    df["timestamp_utc"]    = iso_utc
    df["timestamp_local"]  = iso_local

    # write out
    if not output_path:
        base, ext = os.path.splitext(input_path)
        output_path = f"{base}.timestamp_fixed.csv"
    df.to_csv(output_path, index=False)

    # tiny report
    parsed_ok = int((~ts_utc.isna()).sum())
    print(f"[ok] rows={len(df)} parsed={parsed_ok} saved={output_path}")
    return output_path

# ----------------- RUN (edit INPUT_PATH as needed) -----------------
if __name__ == "__main__":
    # Example: set to your combined file
    INPUT_PATH = r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined\episodes_enriched_combined.csv"
    # If you’re running this next to the CSV, you can also use:
    # INPUT_PATH = "episodes_enriched_combined.csv"
    fix_timestamps(INPUT_PATH)


C:\Users\gilla\AppData\Local\Temp\ipykernel_13072\3677347822.py:80: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  unix_s = ts_utc.view("int64") // 10**9


[ok] rows=19361 parsed=19361 saved=C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined\episodes_enriched_combined.timestamp_fixed.csv


In [13]:
import os
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

# -----------------------------
# Paths
# -----------------------------
INPUT_DIR  = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0")
INPUT_CSV  = INPUT_DIR / "episodes_enriched_combined.csv"          # change if your file name differs
OUTPUT_CSV = INPUT_DIR / "episodes_enriched_combined_expanded.csv"

# -----------------------------
# Helpers: parsing and math
# -----------------------------
def _try_json_load(v: Any):
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, (list, dict)):
        return v
    s = str(v).strip()
    if not s:
        return None
    try:
        return json.loads(s)
    except Exception:
        return None

def _as_list(v: Any) -> List[Any]:
    x = _try_json_load(v)
    return x if isinstance(x, list) else []

def _as_dict(v: Any) -> Dict[str, Any]:
    x = _try_json_load(v)
    return x if isinstance(x, dict) else {}

def _as_int(v: Any) -> Optional[int]:
    try:
        s = str(v).strip()
        if s == "" or s.lower() in {"none", "null", "nan"}:
            return None
        return int(float(s))
    except Exception:
        return None

def _sum_timeouts_seconds(d: Dict[str, Any]) -> Optional[int]:
    if not isinstance(d, dict):
        return None
    total = 0
    found = False
    for val in d.values():
        try:
            text = str(val).strip().lower()
            if text.endswith("ms"):
                n = float(text[:-2]) / 1000.0
            elif text.endswith("s"):
                n = float(text[:-1])
            elif text.endswith("m"):
                n = float(text[:-1]) * 60.0
            elif text.endswith("h"):
                n = float(text[:-1]) * 3600.0
            else:
                n = float(text)
            total += int(n)
            found = True
        except Exception:
            pass
    return total if found else None

# -----------------------------
# Hint regexes
# -----------------------------
GMD_HINTS = re.compile(r"(manageddevice|managed test device|pixel\d*api\d+|gradle\s+managed\s+device)", re.I)
DIY_HINTS = re.compile(r"(emulator\s-|avdmanager|sdkmanager|adb\s|create avd|emulator\.bat|qemu)", re.I)

COVERAGE_FIELDS = {"api_levels", "abis", "device_profiles", "system_images"}

# -----------------------------
# Delta-only intention detection (Part 2)
# -----------------------------
def derive_delta_intentions(row: pd.Series) -> List[str]:
    labels = set()

    field         = str(row.get("field", "") or "").strip()
    old_value     = row.get("old_value", "")
    new_value     = row.get("new_value", "")
    added_items   = _as_list(row.get("added_items", ""))
    removed_items = _as_list(row.get("removed_items", ""))

    # Coverage expansion / reduction
    if field in COVERAGE_FIELDS:
        if added_items:
            labels.add("expand_coverage")
        if removed_items:
            labels.add("reduce_coverage")

    # Timeouts tuning and direction
    if field == "timeouts":
        labels.add("timeouts_tuning")
        a_sum = _sum_timeouts_seconds(_as_dict(old_value))
        b_sum = _sum_timeouts_seconds(_as_dict(new_value))
        if a_sum is not None and b_sum is not None:
            if b_sum > a_sum:
                labels.add("flake_mitigation")
            elif b_sum < a_sum:
                labels.add("speed_up_ci")

    # Retries tuning and direction
    if field == "retries":
        labels.add("retries_tuning")
        a = _as_int(old_value)
        b = _as_int(new_value)
        if a is not None and b is not None:
            if b > a:
                labels.add("flake_mitigation")
            elif b < a:
                labels.add("speed_up_ci")

    # Orchestrator change (assume stability intent)
    if field == "orchestrator":
        labels.add("orchestrator_change")
        labels.add("flake_mitigation")

    # Runner OS change (infra tuning)
    if field == "runner_os":
        labels.add("infra_tuning")

    # Invocation hints: GMD/DIY adoption or drop
    if field == "invocation_hints":
        old_list = _as_list(old_value)
        new_list = _as_list(new_value)
        old_s = " ".join(map(str, old_list))
        new_s = " ".join(map(str, new_list))
        had_gmd = bool(GMD_HINTS.search(old_s))
        has_gmd = bool(GMD_HINTS.search(new_s))
        had_diy = bool(DIY_HINTS.search(old_s))
        has_diy = bool(DIY_HINTS.search(new_s))
        if not had_gmd and has_gmd:
            labels.add("adopt_gmd")
        if had_gmd and not has_gmd:
            labels.add("drop_gmd")
        if not had_diy and has_diy:
            labels.add("adopt_diy")

    # Third-party references change
    if field == "thirdparty_refs":
        labels.add("thirdparty_lab")

    # Return sorted unique labels (as list for now; we join when writing to CSV)
    return sorted(labels)

# -----------------------------
# Main
# -----------------------------
def main():
    if not INPUT_CSV.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV}")

    df = pd.read_csv(INPUT_CSV, encoding="utf-8", low_memory=False)

    # Sanity check for required columns
    needed = {"field", "old_value", "new_value"}
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")

    # Ensure optional columns exist to avoid KeyErrors
    for opt in ("added_items", "removed_items"):
        if opt not in df.columns:
            df[opt] = ""

    # Compute delta-only intentions and format as comma-separated text (no brackets)
    df["delta_intentions"] = (
        df.apply(derive_delta_intentions, axis=1)
          .apply(lambda xs: ",".join(xs) if xs else "")
    )

    # Save
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"[ok] Wrote {OUTPUT_CSV} with delta_intentions as comma-separated text")

if __name__ == "__main__":
    main()


[ok] Wrote C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V3.0\episodes_enriched_combined_expanded.csv with delta_intentions as comma-separated text
